# 05e — Reconstruction: where were the toads each year?

The forecast of notebook 05d starts from the presence estimated in 2025. The same model, fitted with all records up to 2025, can also look back: for every cell and every year since the first record of the invasion, what is the probability that toads had already arrived, given the cell's whole record history?

This is a **reconstruction with hindsight**, not a forecast. A cell whose first record came in 2018, in an area where few people look for frogs, probably had toads a few years earlier, and the model uses that later record to say so. The honest test of the model's predictive skill is the backtest of notebooks 05d and 07, where the model never sees the years it predicts.

For a cell with first record in year *f*, the posterior weight of each possible arrival year *a* ≤ *f* is P(arrive in *a*) × P(missed from *a* to *f* − 1) × P(detected in *f*), divided by the likelihood of the cell; for a cell without records, the weights are P(arrive in *a*) × P(missed from *a* to 2025) divided by its likelihood. The probability of presence by year *t* is the sum of the weights of the arrival years up to *t*.

The definitions (data, detection and arrival models) are the same as in notebook 05d.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.special import expit, log_expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))
from colonization import FEATURES, cell_year_table, simulate
from environment import BIOCLIM_VARIABLES, environmental_grid
from front import records_to_cells
from occupancy import ArrivalLikelihood, counts_per_cell_year, fit_detection, simulate_presence, value_at

PROCESSED, RAW = PROJECT_ROOT / "data" / "processed", PROJECT_ROOT / "data" / "raw"
FIGURES = PROJECT_ROOT / "figures"
CELL_SIZE_M = 25_000
CELL_KM2 = (CELL_SIZE_M / 1000) ** 2
CLIMATE = list(BIOCLIM_VARIABLES.values())
TRAIN_START = 1990
N_BOOTSTRAP = 30
BLOCK_CELLS = 10  # 10 x 25 km = 250 km blocks

cells = environmental_grid(CELL_SIZE_M)
toads = pd.read_csv(PROCESSED / "rhinella_marina_clean.csv")
cells = cells.merge(records_to_cells(toads, CELL_SIZE_M)[["cx", "cy", "first_year"]], how="left", on=["cx", "cy"])
cells["first_year"] = cells["first_year"].fillna(np.inf)
cells["x_km"] = (cells["cx"] + 0.5) * CELL_SIZE_M / 1000
cells["y_km"] = (cells["cy"] + 0.5) * CELL_SIZE_M / 1000
first_year = cells["first_year"].to_numpy()
x_km = cells["x_km"].to_numpy()
west = (cells["lon"] < 141).to_numpy()
outside = (cells["lon"] < 138).astype(int).to_numpy()
desert = (cells["climate_zone"] == "Desert (BW)").to_numpy()
pilbara = (cells["lon"].between(116, 121) & cells["lat"].between(-23, -19)).to_numpy()

amphibians = pd.read_csv(RAW / "amphibia_ala.csv", low_memory=False)
others = amphibians.loc[
    (amphibians["scientificName"] != "Rhinella marina")
    & amphibians["decimalLatitude"].notna() & amphibians["year"].notna()
    & ~(amphibians["coordinateUncertaintyInMeters"] > 10_000)
]
effort = counts_per_cell_year(others, cells, CELL_SIZE_M)
toad_counts = counts_per_cell_year(toads.dropna(subset=["year"]), cells, CELL_SIZE_M)
print(f"{len(cells):,} cells; effort records {int(effort.to_numpy().sum()):,}; toad records {int(toad_counts.to_numpy().sum()):,}")

12,312 cells; effort records 1,843,125; toad records 34,607


In [2]:
REGION = ["outside_queensland", "outside_x_log_distance", "outside_x_neighbours_50km"]
ARRIVAL = FEATURES + REGION
DETECTION = ["log_effort", "outside_queensland"]


def region_terms(table: pd.DataFrame, cell_ids: np.ndarray) -> pd.DataFrame:
    flag = outside[cell_ids]
    table["outside_queensland"] = flag
    table["outside_x_log_distance"] = flag * table["log_distance_km"].to_numpy()
    table["outside_x_neighbours_50km"] = flag * table["neighbours_50km"].to_numpy()
    return table


def detection_table(cutoff: int) -> pd.DataFrame:
    '''Cell-years after each cell's first record, up to the cut-off: toads are there, were they recorded?'''
    occupied = np.flatnonzero(first_year < cutoff)
    start = np.maximum(first_year[occupied].astype(int) + 1, TRAIN_START)
    lengths = np.clip(cutoff - start + 1, 0, None)
    cell_ids = np.repeat(occupied, lengths)
    years = np.concatenate([np.arange(s, cutoff + 1) for s in start])
    table = pd.DataFrame({"cell": cell_ids, "year": years})
    table["log_effort"] = np.log1p(value_at(effort, cell_ids, years))
    table["outside_queensland"] = outside[cell_ids]
    table["detected"] = (value_at(toad_counts, cell_ids, years) > 0).astype(int)
    return table


In [3]:
def suitability(cutoff: int) -> np.ndarray:
    model = make_pipeline(StandardScaler(), PolynomialFeatures(degree=2), LogisticRegression(max_iter=5000))
    return model.fit(cells[CLIMATE], (cells["first_year"] <= cutoff).astype(int)).predict_proba(cells[CLIMATE])[:, 1]


def arrival_table(cutoff: int, score: np.ndarray) -> pd.DataFrame:
    table = cell_year_table(cells, score, range(TRAIN_START, cutoff + 1))
    table["log_effort"] = np.log1p(value_at(effort, table["cell"].to_numpy(), table["year"].to_numpy()))
    return region_terms(table, table["cell"].to_numpy())


def with_detection(table: pd.DataFrame, fit) -> pd.DataFrame:
    eta = fit.linear(table)
    return table.assign(log_detection=log_expit(eta), log1m_detection=log_expit(-eta))


def fit_both(cutoff: int, table: pd.DataFrame, cell_weights: pd.Series | None = None):
    detection = detection_table(cutoff)
    row_weights = None if cell_weights is None else cell_weights.reindex(detection["cell"]).fillna(0).to_numpy()
    detection_model = fit_detection(detection, DETECTION, row_weights)
    likelihood = ArrivalLikelihood(with_detection(table, detection_model), ARRIVAL, cell_weights)
    return likelihood.fit(), detection_model, likelihood


## 1. Fit with all data up to 2025 and reconstruct

In [4]:
FIRST_YEAR = int(first_year[np.isfinite(first_year)].min())
YEARS = range(FIRST_YEAR, 2026)
score = suitability(2025)
table = arrival_table(2025, score)
arrival_fit, detection_fit, likelihood = fit_both(2025, table)

# The coefficients were estimated on 1990-2025; the reconstruction applies them to the whole history since the
# first record, with the same scaling, so the earlier decades are covered by the same model.
# In the release year no cell was occupied before, so the model starts the following year; in the release year,
# presence is exactly the cells with records.
full_table = cell_year_table(cells, score, range(FIRST_YEAR + 1, 2026))
full_table["log_effort"] = np.log1p(value_at(effort, full_table["cell"].to_numpy(), full_table["year"].to_numpy()))
full_table = with_detection(region_terms(full_table, full_table["cell"].to_numpy()), detection_fit)
full_likelihood = ArrivalLikelihood(full_table, ARRIVAL, standardise_like=arrival_fit)
reconstruction = full_likelihood.presence_by_year(arrival_fit, range(FIRST_YEAR + 1, 2026)).reindex(range(len(cells)))
reconstruction.loc[first_year <= FIRST_YEAR] = 1.0
reconstruction.insert(0, FIRST_YEAR, (first_year <= FIRST_YEAR).astype(float))
assert not reconstruction.isna().any().any()
print(f"{len(reconstruction):,} cells × {len(YEARS)} years, {FIRST_YEAR}–2025")

12,312 cells × 91 years, 1935–2025


## 2. Consistency checks

In [5]:
posterior = likelihood.presence_posterior(arrival_fit).reindex(range(len(cells))).fillna(1.0).to_numpy()
checks = {
    "2025: expected cells with toads, reconstruction vs notebook 05d": f"{reconstruction[2025].sum():.0f} vs {posterior.sum():.0f}",
    "2025: mean absolute difference per cell": f"{np.abs(reconstruction[2025].to_numpy() - posterior).mean():.4f}",
    "never decreases over time (toads stay)": bool((reconstruction.diff(axis=1).iloc[:, 1:] >= -1e-9).all().all()),
    "recorded cells are present from their first record": bool(all(
        reconstruction.loc[i, int(first_year[i])] > 0.999 for i in np.flatnonzero(first_year <= 2025)
    )),
}
checks

{'2025: expected cells with toads, reconstruction vs notebook 05d': '1901 vs 1849',
 '2025: mean absolute difference per cell': '0.0042',
 'never decreases over time (toads stay)': True,
 'recorded cells are present from their first record': True}

In [6]:
CELL_KM2 = (CELL_SIZE_M / 1000) ** 2
area = pd.DataFrame({
    "year": list(YEARS),
    "area with records (thousand km²)": [(first_year <= y).sum() * CELL_KM2 / 1000 for y in YEARS],
    "expected area with toads (thousand km²)": [reconstruction[y].sum() * CELL_KM2 / 1000 for y in YEARS],
})
area["unrecorded share of occupied area"] = 1 - area.iloc[:, 1] / area.iloc[:, 2]
area.iloc[::10].round(2)

,year,area with records (thousand km²),expected area with toads (thousand km²),unrecorded share of occupied area
0,1935,0.62,0.62,0.00
10,1945,27.50,91.56,0.70
20,1955,63.75,174.81,0.64
30,1965,108.12,269.18,0.60
40,1975,241.88,382.30,0.37
50,1985,308.12,479.72,0.36
60,1995,365.62,586.87,0.38
70,2005,465.62,748.99,0.38
80,2015,610.00,950.62,0.36
90,2025,805.62,1187.94,0.32


How to read it: the share of the occupied area without records shows how far records lagged behind the toads in each decade. The early decades rest on few records and little survey effort, and they apply coefficients estimated on 1990–2025, so they are the least certain part of the reconstruction. In 2025 the reconstruction differs slightly from the starting point of the forecast (notebook 05d), because here every cell's history starts at the first record of the invasion instead of 1990.

In [7]:
cell_columns = cells[["cx", "cy", "lon", "lat", "first_year"]].reset_index(drop=True)
output = pd.concat([cell_columns, reconstruction.reset_index(drop=True).add_prefix("p_")], axis=1)
output.to_csv(PROCESSED / "reconstruction_presence_model.csv", index=False)
print("Saved reconstruction_presence_model.csv")

Saved reconstruction_presence_model.csv
